## Dataset: CAMELYON17

[CAMELYON17](https://camelyon17.grand-challenge.org/) is a multi-center histopathology dataset for detecting breast cancer metastases in lymph node whole-slide images (WSIs). It extends CAMELYON16 to 5 medical centers, adding a stronger focus on inter-center variability (staining, scanners, protocols) to test model generalization.

- **`metadata.csv`** — per-sample metadata (e.g. patient/node identifiers, center, slide-level label, and any train/test split info). Use this as the index for loading and joining samples.
- **`patches/`** — extracted image patches (tiles) sampled from the original WSIs, organized by class/label, used for patch-level classification instead of loading full slides.

You can load `metadata.csv` to get an overview of class balance and center distribution, then sample a few images from `patches/` to visualize.

In [ ]:
import matplotlib.pyplot as plt
import os
import pandas as pd
from pathlib import Path
from glob import glob

In [ ]:
camelyon_base_dir = "/home/shared/data/camelyon17/camelyon17_v1.0/"
metadata_path = os.path.join(camelyon_base_dir, "metadata.csv")

In [ ]:
sample_img_path = glob(camelyon_base_dir + "**/*.png", recursive=True)[0]
plt.imshow(plt.imread(sample_img_path))


In [ ]:
metadata_df = (
    pd.read_csv(metadata_path, index_col=False).drop("Unnamed: 0", axis=1)
    # .groupby("tumor").sample(10, random_state=42)
    .assign(patient_node=lambda df_: df_.apply(lambda row: f"patient_{row['patient']:03d}_node_{row['node']}", axis=1))
    .assign(filepath=lambda df_: df_.apply(lambda row: Path(f"{row['patient_node']}/patch_{row['patient_node']}_x_{row['x_coord']}_y_{row['y_coord']}.png"), axis=1))
    .assign(filepath_abs=lambda df_: df_["filepath"].apply(lambda p: Path(camelyon_base_dir) / "patches" / p))
    .drop("patient_node", axis=1)
)
metadata_df

In [ ]:
# plot a 3x6 grid with left side tumor 0 and green spines, right side tumor 1 and red spines, and the title of each subplot is the patient number

fig, axs = plt.subplots(2, 6, figsize=(12, 6))
for i, (tumor, group) in enumerate(metadata_df.groupby("tumor")):
    ax_row = axs[i]
    for j, (patient, patient_group) in enumerate(group.groupby("patient")):
        if j > 5:
            break
        ax = ax_row[j]
        sample_img_path = patient_group["filepath_abs"].iloc[0]
        ax.imshow(plt.imread(sample_img_path))
        ax.set_title(f"Patient {patient}")
        ax.set_xticks([])
        ax.set_yticks([])
        if tumor == 0:
            for spine in ax.spines.values():
                spine.set_edgecolor("green")
                spine.set_linewidth(2)
        else:
            for spine in ax.spines.values():
                spine.set_edgecolor("red")
                spine.set_linewidth(2)

# add legend for tumor 0 and tumor 1
fig.legend(
    handles=[
        plt.Line2D([0], [0], color="green", lw=2, label="Tumor=0"),
        plt.Line2D([0], [0], color="red", lw=2, label="Tumor=1"),
    ],
    loc="center",
    bbox_to_anchor=(0.5, 0.5),
    ncol=2,
)

plt.tight_layout()


### Center (hospital) comparison

Redo the patient grid above but group by `center` instead of `patient` — CAMELYON17's main challenge is inter-center variability (staining color, scanner), so this is the most visually striking axis to explore.

### Tumor prevalence per center

Does tumor rate vary a lot by hospital, or is it fairly balanced?

### Split distribution

See which centers are held out for val/test (WILDS-style splits typically leave whole centers out for OOD evaluation).

### Patch counts

Check for imbalance in how many patches come from each slide/patient.

### Spatial layout of one slide

Pick a single `(patient, node)`, scatter `x_coord` vs `y_coord` colored by `tumor` — tumor patches should cluster spatially since they come from real tumor regions on the slide.

### Color/staining stats

Compute mean RGB (or HSV) per patch and compare distributions across `center` — a quick proxy for the staining differences models need to generalize across.

### Zoom into a slide

Patches from the same `(patient, node)` tile a region of the WSI — plotting several neighboring patches (close `x_coord`/`y_coord`) side by side gives a sense of local tissue context.

### Library to access CAMELYON data
There is also a library to make data handling easier. If you want you can explore it more:

In [ ]:

from wilds import get_dataset

CAMELYON17_DIR = "/home/shared/data/camelyon17"
dataset = get_dataset(dataset="camelyon17", download=True, root_dir=str(CAMELYON17_DIR))

print("Dataset:", dataset)
print("Num examples:", len(dataset))
print("Metadata fields:", dataset.metadata_fields)  # includes hospital / center id
print("\nSaved to:", CAMELYON17_DIR)